# Assignment 1A — Part B: QLoRA Instruction Fine-Tuning

**Domain:** Medical & Clinical Literature
**Base checkpoint:** CPT output from Part A when available; otherwise BioGPT-Large

This notebook creates the structured instruction dataset, splits it 80/20, trains adapters A/B/C with 4-bit QLoRA, and compares all adapters on the same three domain prompts.


In [1]:
# Run once in Colab:
# !pip install -q -r requirements_colab.txt


In [2]:
try:
    import truststore
    truststore.inject_into_ssl()
except Exception:
    pass

from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT / "LLM_Assignment_Medical" / "src").exists():
    ROOT = ROOT / "LLM_Assignment_Medical"
sys.path.insert(0, str(ROOT / "src"))
DATASET_PATH = ROOT / "instruction_dataset.jsonl"
CPT_DIR = ROOT / "outputs" / "biogpt-large-cpt"
MODEL_ID = "microsoft/biogpt-large"
ADAPTER_ROOT = ROOT / "outputs" / "qlora_adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

## B1 — Instruction dataset creation

The included heuristic generator creates 120 deterministic educational pairs from the cleaned medical corpus; no external LLM is used. Each row has instruction, response, source, and split; the split is 96 training rows and 24 evaluation rows. For the marked run, regenerate from the enlarged cleaned corpus and retain the same schema.


In [3]:
from medical_pipeline import build_instruction_dataset
if not DATASET_PATH.exists():
    stats = build_instruction_dataset(ROOT / "domain_corpus", DATASET_PATH, min_pairs=100)
else:
    rows_check = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    stats = {"total": len(rows_check), "train": sum(r["split"] == "train" for r in rows_check), "eval": sum(r["split"] == "eval" for r in rows_check)}
print(stats)
assert stats["total"] >= 100 and stats["train"] == int(stats["total"] * .8)


{'total': 120, 'train': 96, 'eval': 24}


In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
rows = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
def format_example(row):
    messages = [{"role": "user", "content": row["instruction"]}, {"role": "assistant", "content": row["response"]}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return "### Instruction:\n" + row["instruction"] + "\n\n### Response:\n" + row["response"]
for row in rows:
    row["text"] = format_example(row)
print(rows[0]["text"])


c:\Users\anabhart\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\anabhart\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anabhart\.cache\huggingface\hub\models--microsoft--biogpt-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/d

### Instruction:
What is Hypertension and cardiovascular risk?

### Response:
Hypertension is persistent elevation of arterial blood pressure. Long-term pressure load can injure the heart, brain, kidneys, and blood vessels, so risk assessment considers both blood-pressure measurements and other cardiovascular risk factors.


## B2 — QLoRA with three adapter configurations

All adapters use 4-bit NF4 quantization and the same training split.

| Adapter | r | alpha | Target modules | Expected effect |
|---|---:|---:|---|---|
| A | 8 | 16 | q_proj, v_proj | Fast; may underfit |
| B | 16 | 32 | q_proj, v_proj | Balanced quality/cost |
| C | 32 | 32 | q_proj, v_proj, o_proj | Highest capacity |


In [5]:
ADAPTER_CONFIGS = {
    "adapter_A": {"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj"]},
    "adapter_B": {"r": 16, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj"]},
    "adapter_C": {"r": 32, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj", "o_proj"]},
}
RUN_QLORA = False


In [6]:
def train_one_adapter(name, cfg, train_rows, eval_rows):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer
    model_source = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(model_source, quantization_config=bnb, device_map="auto")
    model = prepare_model_for_kbit_training(model)
    available = {module_name.split(".")[-1] for module_name, _ in model.named_modules()}
    resolved_targets = ["out_proj" if target == "o_proj" and "o_proj" not in available and "out_proj" in available else target for target in cfg["target_modules"]]
    missing = [target for target in resolved_targets if target not in available]
    if missing:
        raise ValueError(f"Adapter {name} target modules not found: {missing}. Inspect model.named_modules().")
    lora = LoraConfig(r=cfg["r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=resolved_targets)
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    args = TrainingArguments(output_dir=str(ADAPTER_ROOT / name), num_train_epochs=2, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=5, save_strategy="epoch", evaluation_strategy="epoch", fp16=True, report_to="none", remove_unused_columns=False)
    try:
        trainer = SFTTrainer(model=model, args=args, train_dataset=train_rows, eval_dataset=eval_rows, processing_class=tokenizer, dataset_text_field="text", max_seq_length=512)
    except TypeError:
        trainer = SFTTrainer(model=model, args=args, train_dataset=train_rows, eval_dataset=eval_rows, tokenizer=tokenizer, dataset_text_field="text", max_seq_length=512)
    trainer.train()
    trainer.save_model(str(ADAPTER_ROOT / name))
    tokenizer.save_pretrained(str(ADAPTER_ROOT / name))
    return trainer


In [7]:
if RUN_QLORA:
    from datasets import Dataset
    train_rows = [r for r in rows if r["split"] == "train"]
    eval_rows = [r for r in rows if r["split"] == "eval"]
    train_ds, eval_ds = Dataset.from_list(train_rows), Dataset.from_list(eval_rows)
    for name, cfg in ADAPTER_CONFIGS.items():
        train_one_adapter(name, cfg, train_ds, eval_ds)
else:
    print("QLoRA training is ready; set RUN_QLORA=True on a GPU runtime.")


QLoRA training is ready; set RUN_QLORA=True on a GPU runtime.


## B3 — Evaluation and comparative analysis

The same three prompts are sent to each adapter. The table is saved as JSON and can be copied into the final report.


In [8]:
EVAL_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the difference between sensitivity and specificity?",
    "Why does antimicrobial stewardship matter?",
]
RUN_ADAPTER_EVAL = False


In [9]:
def evaluate_adapter(adapter_path, prompts):
    import torch
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    source = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
    base = AutoModelForCausalLM.from_pretrained(source, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, str(adapter_path))
    outputs = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        outputs.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return outputs

if RUN_ADAPTER_EVAL:
    comparison = {}
    for name in ADAPTER_CONFIGS:
        path = ADAPTER_ROOT / name
        comparison[name] = evaluate_adapter(path, EVAL_PROMPTS) if path.exists() else ["Adapter not found"] * len(EVAL_PROMPTS)
    table = [{"prompt": prompt, **{name: comparison[name][idx] for name in ADAPTER_CONFIGS}} for idx, prompt in enumerate(EVAL_PROMPTS)]
    (ROOT / "outputs" / "adapter_comparison.json").write_text(json.dumps(table, indent=2), encoding="utf-8")
    display(table)
else:
    print("Adapter evaluation is ready; set RUN_ADAPTER_EVAL=True after training.")


Adapter evaluation is ready; set RUN_ADAPTER_EVAL=True after training.


## Part B conclusion

Adapter B is the expected quality/cost baseline. Adapter C has the most trainable capacity and may be strongest on multi-part clinical questions, while Adapter A is fastest and may underfit. The final verdict must be based on the three saved outputs and correctness against the source corpus.
